In [13]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time

In [14]:
ORG_URL = "https://github.com/orgs/google/repositories"
BASE_URL = "https://github.com"

In [15]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
}

In [ ]:
time.sleep(1)
res = requests.get(ORG_URL, headers=headers, timeout=10)
print("ステータスコード:", res.status_code)
res.raise_for_status()
soup = BeautifulSoup(res.text, "html.parser")

repo_links = []
seen = set()

name_language_stars = {}

for a in soup.find_all("a", href=True):

    href = a["href"]
    # /google/xxx という形だけ
    if href.startswith("/google/") and href.count("/") == 2:
        if href not in seen:
            seen.add(href)
            repo_links.append(a)


for a in repo_links:
    name = a.get_text(strip=True)
    repo_path = a["href"]
    repo_url = BASE_URL + repo_path

    print(f"\n--- {name} を取得中: {repo_url}")
    time.sleep(1)

    r = requests.get(repo_url, headers=headers, timeout=10)
    if r.status_code != 200:
        print("  リポジトリページ取得失敗:", r.status_code)
        continue

    rsoup = BeautifulSoup(r.text, "html.parser")

    language = "Unknown"
    for tag in rsoup.find_all("a"):
        text = tag.get_text(strip=True)
        if text.endswith("%") and " " in text:
            language = text.split()[0]
            break


    stars = "0"
    for tag in rsoup.find_all("a"):
        text = tag.get_text(strip=True)
        if "stars" in text:
            stars = text.split()[0]
            break

    print("SCRAPED:", name, language, stars)
    name_language_stars[name] = (language, stars)
print("見つかったリポジトリ数:", len(repo_links))
print("件数:", len(name_language_stars))
print("例:", list(name_language_stars.items())[:3])


ステータスコード: 200

--- zerocopy を取得中: https://github.com/google/zerocopy
SCRAPED: zerocopy Unknown 2.1kstars

--- skia-buildbot を取得中: https://github.com/google/skia-buildbot
SCRAPED: skia-buildbot Unknown 158stars

--- XNNPACK を取得中: https://github.com/google/XNNPACK
SCRAPED: XNNPACK Unknown 2.2kstars

--- gtm-session-fetcher を取得中: https://github.com/google/gtm-session-fetcher
SCRAPED: gtm-session-fetcher Unknown 265stars

--- google-toolbox-for-mac を取得中: https://github.com/google/google-toolbox-for-mac
SCRAPED: google-toolbox-for-mac Unknown 1.2kstars

--- ground-android を取得中: https://github.com/google/ground-android
SCRAPED: ground-android Unknown 272stars

--- command-fds を取得中: https://github.com/google/command-fds
SCRAPED: command-fds Unknown 42stars

--- yapf を取得中: https://github.com/google/yapf
SCRAPED: yapf Unknown 14kstars

--- angle を取得中: https://github.com/google/angle
SCRAPED: angle Unknown 3.8kstars

--- tsl を取得中: https://github.com/google/tsl
SCRAPED: tsl Unknown 101stars

--- 

In [29]:
path = ''
db_name = 'test.db'

try:


    # DB接続オブジェクトの作成
    conn = sqlite3.connect(path + db_name)

    # SQL（RDBを操作するための言語）を実行するためのカーソルオブジェクトを取得
    cur = conn.cursor()
    # SQL文の作成（テーブルの作成）
    sql = 'CREATE TABLE google (name TEXT, language TEXT, stars INT  );'

    # SQLの実行
    cur.execute(sql)
    conn.commit()
    print("テーブル作成OK")
    
except sqlite3.Error as e:
    print(f"エラーが発生しました: {e}")

finally:
    # DBへの接続を閉じる
    conn.close()

テーブル作成OK


In [30]:
path = ''
db_name = 'test.db'

try:
    conn = sqlite3.connect(path + db_name)
    cur = conn.cursor()

    sql = "INSERT INTO google (name, language, stars) VALUES (?, ?, ?)"

    for name, (language, stars) in name_language_stars.items():
        cur.execute(sql, (name, language, stars))

    conn.commit()
    print("INSERT 完了")

except sqlite3.Error as e:
    print(f"エラーが発生しました: {e}")

finally:
    conn.close()


INSERT 完了


In [32]:
path = ''
db_name = 'test.db'

try:
    # DB接続オブジェクトの作成
    conn = sqlite3.connect(path + db_name)

    # SQLを実行するためのカーソルオブジェクトを取得
    cur = conn.cursor()

    # データの取得
    sql = "SELECT * FROM google"

    # SQLの実行
    cur.execute(sql)

except sqlite3.Error as e:
    print(f"エラーが発生しました: {e}")

else:
    # ここで取得したデータを表示
    # テーブルは (name, language, stars) の3カラムなので、それに合わせてアンパックします。
    for idx, row in enumerate(cur, start=1):
        name, language, stars = row
        print(idx, name, language, stars)

finally:
    # DBへの接続を閉じる
    conn.close()


1 zerocopy Unknown 2.1kstars
2 skia-buildbot Unknown 158stars
3 XNNPACK Unknown 2.2kstars
4 gtm-session-fetcher Unknown 265stars
5 google-toolbox-for-mac Unknown 1.2kstars
6 ground-android Unknown 272stars
7 command-fds Unknown 42stars
8 yapf Unknown 14kstars
9 angle Unknown 3.8kstars
10 tsl Unknown 101stars
11 osv-scalibr Unknown 537stars
12 synopsys-dw-uart Unknown 0
13 cameratrapai Jupyter 386stars
14 device-infra Unknown 58stars
15 double-conversion Unknown 1.2kstars
16 capslock Unknown 1.1kstars
17 dive Unknown 17stars
18 tunix Unknown 1.9kstars
19 adk-samples Jupyter 6.4kstars
20 nearby Unknown 887stars
